# REST Function-as-Table — Unified Sourcing + Action Surface

**Design epic:** `bd-824z`  
**Plan id:** `rest-fn-table-2026-06-04`  
**Date:** 2026-06-04  
**Status:** draft — awaiting user review

**One line:** Make the REST table gateway a *generic* "REST API function as a DuckDB table" for analyst sourcing **and** reusable for actions (writes/RPC), including ad-hoc Google Ads / Facebook Ads-style APIs — delivered as a 3-phase program that is mostly **additive wiring**, not an engine rewrite.

**Overview — one manifest, three SQL-visible projections, one analyst surface:**

```mermaid
flowchart LR
    SRC["Any REST API<br/>(GAds, FB Ads, GitHub, ...)"] --> M["One Manifest<br/>= a 'REST function'"]
    M --> T["table<br/>SELECT * FROM src_t"]
    M --> TF["table function<br/>SELECT * FROM src_tf('arg')"]
    M --> AC["action<br/>SELECT * FROM src_act('arg')<br/>or invoke (write)"]
    T --> AN((Analyst SQL))
    TF --> AN
    AC --> AN
```

## 1. Motivation

An analyst should be able to point at any REST endpoint and immediately `SELECT` from it, and call the same definition as an action when it mutates. Today the gateway *almost* does this but the capability is not reachable:

- A plain `[[table]]` is queryable (`GET` + predicate pushdown), but a parameterized or `POST`-bodied call (Google Ads GAQL `search`, Facebook insights) is modeled as an `[[action]]` — which returns rows from the engine but **cannot be `SELECT`ed**.
- Curated `tier-a/*.connection.toml` presets exist but are **never loaded by production code** (test-only); the Nango picker always emits a read-only, action-less stub.

The goal is one coherent primitive — a *REST function* — projected as a table (read), an arg-taking table function (parameterized read), or an action (invoke), all from one manifest definition, reachable from SQL and from the picker/ad-hoc authoring.

## 2. Grounding (what the code already does)

Verified against the code graph + spur-analyst DuckDB index (artifact hash `17c0db51…`, matches `code_*` responses).

**The three-projection model already exists in the type system** — `TableKind` in `crates/spur-notebook/rest-table-gateway/src/adapter/mod.rs`:

```rust
pub enum TableKind {
    Table,
    TableFunction { arg_names: Vec<String> },
    Action { method: String, path: String, arg_specs: Vec<ArgSpec>,
             dry_run_arg: Option<String>, idempotency_header: Option<String> },
}
```

**All three projections already return `Vec<RecordBatch>`:**

| Projection | Type exists? | Engine returns rows? | Wired to SQL? |
|---|---|---|---|
| `Table` (GET + predicate pushdown) | yes | yes — `scan()` | **yes** — registered |
| `TableFunction { arg_names }` (GET **with args**) | yes — variant + `ScanRequest.tvf_args` field | partial — `scan()` ignores `tvf_args` | **no** — skipped in `register_tables`; `parameters()` → `None`; `init` passes `tvf_args: vec![]` |
| `Action { method, path, arg_specs, … }` (any method, body templating, **`columns`**, **`pagination`**) | yes | yes — `act()` + `call_act` bridge | **no** — skipped in `register_tables`; only a test calls `call_act` |

**Wiring status at a glance** (green = live, orange = half-scaffolded, red = built-but-unreached):

```mermaid
flowchart LR
    M["Manifest definition"]
    M --> T["Table<br/>GET + pushdown"]
    M --> TF["TableFunction<br/>{ arg_names }"]
    M --> AC["Action<br/>method · columns · pagination"]
    T -->|"scan() → rows"| REG{{"register_tables"}}
    TF -. "scan() ignores tvf_args" .-> REG
    AC -. "act()/call_act → rows · only a test calls call_act" .-> REG
    REG ==>|"only Table registered today"| SQL[("DuckDB SQL")]
    classDef wired fill:#1b5e20,color:#fff;
    classDef partial fill:#e65100,color:#fff;
    classDef unwired fill:#b71c1c,color:#fff;
    class T wired
    class TF partial
    class AC unwired
```

Key evidence:
- `ActionCfg` (`adapter/manifest.rs`) already carries `columns: Option<…>` and `pagination: Option<ActionPaginationCfg>` — `act()` is a typed, paginated, row-returning engine.
- `IoBridge::call_act` (`vtab/bridge.rs`) bridges actions to rows and is `pub` — but `call_act` has **exactly one caller, a test** (`bridge_dispatches_act`). The action→rows engine is *built but unreached from SQL*.
- `register_tables` (`vtab/register.rs`) is the single registration chokepoint and hard-skips everything that is not `TableKind::Table`: `if !matches!(t.kind, TableKind::Table) { continue; }`.
- `ApiTableVTab::parameters()` returns `None` and `ApiTableVTab::init` builds `ScanRequest { …, tvf_args: vec![], … }` — the arg plumbing exists in the request type but is never populated.

## 3. Blast radius (why this is safe)

From `v_blast_radius` + caller breakdown (spur-analyst):

- `register_tables` — 3 callers, **all tests**. Body change (also register `TableFunction`/`Action`) needs no signature change.
- `call_act` — 1 caller, **a test**. Wiring it into registration is net-additive; no existing production consumer to regress.
- `provider_to_manifest_stub` — 10 callers, only **2 production** (`build_api_import_manifest`, the `nango-import` CLI). Phase-3 Nango bridge touches 1 production site.
- `scan` / `act` / `catalog` show 0 resolved callers because they are invoked through `Arc<dyn Adapter>` dynamic dispatch (DuckPGQ does not capture dyn edges) — so changes to their bodies do not ripple through resolved call sites.

**Conclusion:** the work is contained to `vtab/register.rs`, `vtab/table_fn.rs` (+ a possible new `vtab/action_fn.rs`), and the catalog mapping in `manifest_adapter.rs`. No engine rewrite.

## 4. Architecture — the unified REST-function model

One manifest, one `Adapter`, three SQL-visible projections of the same definition. The engine and the IO bridge are already shared; unification happens at the **registration + VTab arg-binding** layer.

```mermaid
flowchart TD
    M["Manifest<br/>source: auth · headers · pagination · base_url · allow_writes<br/>tables[] · actions[]"]
    M -->|"Adapter::catalog()"| CAT["Vec of CatalogEntry { name, kind, schema }"]
    CAT --> REG{{"register_tables(conn, adapter, bridge)"}}
    REG -->|"TableKind::Table"| VT1["ApiTableVTab (no args)"]
    REG -->|"TableKind::TableFunction · Phase 2"| VT2["ApiTableVTab + parameters() (named args)"]
    REG -->|"TableKind::Action(cols) · Phase 1"| VT3["ApiActionVTab + parameters() (action args)"]
    VT1 -->|"bridge.call(Scan)"| BR{{"IoBridge — single rest-gateway-io thread"}}
    VT2 -->|"bridge.call(Scan { tvf_args })"| BR
    VT3 -->|"bridge.call_act(Action { args })"| BR
    BR -->|"Job::Scan"| SCAN["Adapter::scan() → rows"]
    BR -->|"Job::Act"| ACT["Adapter::act() → rows"]
    SCAN --> OUT[("DuckDB rows")]
    ACT --> OUT
    classDef p1 fill:#0d47a1,color:#fff;
    classDef p2 fill:#4a148c,color:#fff;
    class VT3 p1
    class VT2 p2
```

Projection is chosen at **definition** time (manifest `[[table]]` vs `[[action]]` + presence of `args`/`columns`) and surfaced at **call** time (`SELECT * FROM src_table` vs `SELECT * FROM src_action('arg')`). A mutating action keeps fire-and-forget semantics behind the `allow_writes` gate; a row-returning action (GAQL/insights) is a read-only `POST` projected as a table function.

## 5. Phase 1 — Action as a DuckDB table function (the headline; unblocks GAds/FB Ads reads)

**Intent:** make a row-returning `Action` (one that declares `columns`) `SELECT`-able with arguments, reusing the already-built `act()` + `call_act`.

**Changes:**
1. **`vtab/register.rs` — `register_tables`:** stop skipping `TableKind::Action`. For an action entry that declares `columns`, register an arg-taking DuckDB table function named `{adapter.name()}_{action.name}`.
2. **`vtab/action_fn.rs` (new) — `ApiActionVTab`:** mirror `ApiTableVTab` but
   - `parameters()` returns the action's positional arg types (from `arg_specs`),
   - `bind` reads `bind.get_parameter(i)` → builds an `ActionRequest { name, method, path, query/body from args, dry_run:false, … }`,
   - `init` calls `bridge.call_act(adapter, req)` and buffers the returned batches,
   - `func` streams rows exactly like `ApiTableVTab::func` (reuse `write_batch_rows`).
3. **`allow_writes` gate:** the existing guard in `act()` (first statement) stays authoritative. Row-returning read POSTs (GAQL/insights) run with `allow_writes=true` in their preset because the upstream endpoint is a POST; for **mutating** actions the registration may additionally annotate the function so a pure-read analyst surface does not silently mutate. (See §11, open question 2.)

**Data flow (Google Ads GAQL):**

```mermaid
sequenceDiagram
    participant SQL as DuckDB SQL
    participant V as ApiActionVTab
    participant B as IoBridge
    participant A as Adapter::act()
    participant API as REST endpoint
    SQL->>V: SELECT * FROM gads_google_ads_search('SELECT ...')
    V->>V: bind() — get_parameter(i) builds ActionRequest
    V->>B: call_act(adapter, req)
    B->>A: Job::Act on rest-gateway-io thread
    A->>A: allow_writes gate · resolve_auth · resolve_headers
    A->>API: POST body = GAQL (+ action pagination loop)
    API-->>A: JSON rows + nextPageToken
    A-->>B: Vec of RecordBatch
    B-->>V: Vec of RecordBatch
    V-->>SQL: streamed rows via func()
```

**Files:** `vtab/register.rs`, new `vtab/action_fn.rs`, `vtab/mod.rs` (module decl), small helper sharing with `vtab/table_fn.rs`. **No** changes to `act()` engine.

## 6. Phase 2 — Finish the `TableFunction { arg_names }` GET-TVF (parameterized reads)

**Intent:** arg-parameterized GET tables, e.g. `gh_repos('owner','name')`, completing the projection that is half-scaffolded.

**Changes:**
1. **`ApiTableVTab`:** `parameters()` returns arg types when the catalog entry is `TableFunction`; `bind`/`init` populate `ScanRequest.tvf_args` (today hardcoded `vec![]`).
2. **`scan()` (`manifest_adapter.rs`):** thread `tvf_args` into path templating + `query_params` (named substitution mirroring action arg substitution; reuse the placeholder guard from the ext crate).
3. **`register_tables`:** register `TableKind::TableFunction` via `ApiTableVTab` with parameters.

```mermaid
flowchart LR
    SQL["SELECT * FROM gh_repos('owner','name')"] --> BIND["ApiTableVTab::bind<br/>get_parameter → tvf_args"]
    BIND --> SR["ScanRequest { tvf_args }"]
    SR --> SCAN["scan(): substitute tvf_args<br/>into path + query_params"]
    SCAN --> GUARD{{"placeholder guard (ext crate)"}}
    GUARD --> FETCH["fetch_rows → rows"]
    FETCH --> SQL
```

**Files:** `vtab/register.rs`, `vtab/table_fn.rs`, `adapter/manifest_adapter.rs` (`scan` + helpers). Depends on the arg-binding helper introduced in Phase 1 (shared).

## 7. Phase 3 — Catalog unification + ad-hoc authoring (the "generic tool")

**Intent:** every way in produces a complete REST function; analysts can stand up Google Ads / Facebook Ads (and arbitrary APIs) without a hand-shipped preset.

**Changes:**
1. **Curated-preset bridge:** `build_api_import_manifest` (`src/mcp/mod.rs`) prefers a bundled `connections/tier-a/<provider>.connection.toml` when present, else falls back to `provider_to_manifest_stub`. Handle the `-`→`_` filename normalization (snapshot keys are hyphenated, e.g. `google-ads`; preset files use `_`, e.g. `google_ads.connection.toml`; `normalize_api_datasource_source` only lowercases/trims). This lights up *all 13* tier-a presets in the picker, not just GAds.
2. **Snapshot:** add `google-ads` and `facebook-ads` (tier B / OAuth2) to `nango_providers_snapshot.yaml` so they appear in the picker.
3. **New presets:** `connections/tier-a/google_ads.connection.toml` (exists) + `facebook_ads.connection.toml` (new) carrying auth + required headers + the row-returning action with `columns`/`pagination`.
4. **Ad-hoc authoring:** `add_api_datasource_from_manifest` already accepts hand-authored manifest TOML — document/expose it as the analyst seam for a runtime-defined REST function (auth + headers + pagination + query/body template + columns).

**Every source converges on the Manifest IR, then the shared registration:**

```mermaid
flowchart TD
    P["Curated preset<br/>tier-a/*.connection.toml"] --> BIM["build_api_import_manifest<br/>prefer preset, else stub"]
    N["Nango provider<br/>snapshot.yaml"] --> BIM
    O["OpenAPI spec_text"] --> BIM
    H["Ad-hoc TOML<br/>add_api_datasource_from_manifest"] --> BIM
    BIM --> IR["Manifest (universal IR)"]
    IR --> REG{{"register_tables"}}
    REG --> SQL[("DuckDB: tables · table-functions · actions")]
    NEW["new: google-ads · facebook-ads<br/>snapshot + presets"] -.-> P
    NEW -.-> N
    classDef new fill:#1b5e20,color:#fff;
    class NEW new
```

**Files:** `src/mcp/mod.rs`, `nango_providers_snapshot.yaml`, new preset TOML(s). Depends on Phase 1 (action-TVF) so picked presets are actually queryable.

## 8. Error handling & security

- **Write gate:** `act()`'s `allow_writes` guard (first statement) is the single authority; Phase 1 does not weaken it. A connection must set `allow_writes = true` for any action, including read-returning POSTs.
- **Placeholder safety:** path/body templating reuses the ext-crate `ensure_no_unfilled_placeholders` (lone `{`/`}` rejected) and `substitute_path_arg` FORBIDDEN set (`/ ? # % \` + `..`). Phase 2 arg substitution must route through the same guards.
- **Secrets:** credentials remain session-scoped env vars (`${connectionConfig.*}` / `SPUR_CONN_*`), intentionally **not** persisted to the notebook catalog — unchanged.
- **Arg typing:** DuckDB function args are validated by `parameters()` logical types before reaching the engine; malformed args fail at bind, not mid-fetch.

## 9. Testing strategy

TDD per SPUR cadence (failing test commit → fix commit). Build/test via `scripts/spur-cargo` (remote default); the **ext crate is a separate workspace** — use `--manifest-path crates/spur-notebook/rest-table-gateway-ext/Cargo.toml`, never `-p`.

- **Phase 1:** unit test that an `Action` with `columns` registers as a table function and `SELECT … FROM <fn>('arg')` returns typed rows (wiremock); test that the `allow_writes=false` gate still errors through the VTab path; e2e mirroring `vtab_e2e.rs`.
- **Phase 2:** test `tvf_args` populate `ScanRequest` and substitute into path/query; placeholder-guard rejection test.
- **Phase 3:** test `build_api_import_manifest` prefers the curated preset (full manifest, not stub) and the `-`→`_` normalization; snapshot-contains-google-ads test; preset-parses tests (pattern of existing `google_ads_preset_has_refresh_auth_and_required_headers`).

## 10. Task decomposition boundaries (for writing-plans)

Each phase becomes its own epic. Within Phase 1 (the immediate plan):

```mermaid
flowchart LR
    T1["T1 · ApiActionVTab<br/>vtab/action_fn.rs<br/>bind/init/func + parameters()"] --> T2["T2 · registration wiring<br/>vtab/register.rs<br/>register Action(cols)"]
    T2 --> T3["T3 · e2e + gate test<br/>SELECT FROM action_fn('arg')<br/>+ allow_writes=false errors"]
    classDef root fill:#0d47a1,color:#fff;
    class T1 root
```

- **T1 — `ApiActionVTab`** (new `vtab/action_fn.rs`): bind/init/func + `parameters()` arg binding + reuse `write_batch_rows`. Depends on: none.
- **T2 — registration wiring** (`vtab/register.rs`): register `TableKind::Action` (cols present) via `ApiActionVTab`. Depends on: T1 (needs the VTab type).
- **T3 — e2e + gate test** (`tests/`): `SELECT … FROM <action_fn>('arg')` typed rows + `allow_writes=false` error path. Depends on: T2.

T1 is the root; T2→T3 chain. Phases 2 and 3 are separate epics, brainstormed/planned after Phase 1 proves the substrate.

## 11. Risks & open questions

1. **DuckDB TVF arg semantics:** confirm the `duckdb` crate binding supports `register_table_function_with_extra_info` + positional `parameters()` reading via `bind.get_parameter(i)` for our version (the `TableFunction` variant implies it was planned). *Resolve in Phase-1 T1 spike.*
2. **Read-POST vs mutating action:** a row-returning POST (GAQL) and a mutating POST (create campaign) both pass through `act()`. Decide whether the analyst table-function surface should be restricted to actions flagged read-only, to avoid a `SELECT` triggering a mutation. Proposal: only register actions whose preset marks them `readonly = true` (or that declare `columns` AND a non-mutating method allow-list) as table functions; keep mutating actions on the existing invoke path.
3. **Naming collisions:** `{adapter}_{name}` must stay unique across tables/table-functions/actions of one source — add a catalog-build assertion.
4. **Pagination parity:** action pagination (`ActionPaginationCfg`) and scan pagination differ; ensure the action-TVF honors action pagination end-to-end (already in `act()`).

## 12. Phasing & next step

Sequence **1 → 2 → 3**, each its own beads epic. Phase 1 has the highest value-per-unit-work: the row-returning action engine is built and unreached, so we are connecting a wire, not laying one — and it directly delivers the Google Ads / Facebook Ads analyst-read use case.

```mermaid
flowchart LR
    P1["Phase 1 · NOW<br/>Action → table function<br/>unblocks GAds/FB reads"] --> P2["Phase 2<br/>TableFunction GET-TVF<br/>parameterized reads"]
    P2 --> P3["Phase 3<br/>catalog unification<br/>+ ad-hoc authoring + presets"]
    classDef now fill:#1b5e20,color:#fff;
    class P1 now
```

**Next:** on approval of this spec, invoke `writing-plans` to produce the Phase-1 beads-backed DAG (T1→T2→T3 above) and `submit_plan` to dispatch.